# Walmart Sales Data – Cleaning & Loading Pipeline

**Goal:** Clean the raw `Walmart.csv` file and load it into **MySQL** and **PostgreSQL**.

**Steps**
1. Import libraries
2. Load data
3. Explore data
4. Clean data (duplicates, missing values, data types, column names)
5. Feature engineering (`total`)
6. Save clean CSV
7. Load into MySQL & PostgreSQL

## 1. Import Libraries

In [ ]:
import os
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, text

# DB drivers (used by SQLAlchemy behind the scenes):
#   pymysql   -> MySQL
#   psycopg2  -> PostgreSQL
import pymysql      # noqa: F401
import psycopg2     # noqa: F401

print(pd.__version__)

## 2. Load Data
`encoding_errors='ignore'` silently drops undecodable characters, so check that no important text was lost.

In [ ]:
df = pd.read_csv('Walmart.csv', encoding_errors='ignore')
df.shape

## 3. Explore Data

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

## 4. Data Cleaning

### 4.1 Standardise column names
Done first so every later step uses the same lowercase names.

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df.columns

### 4.2 Remove duplicate rows

In [ ]:
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicates after :', df.duplicated().sum())
df.shape

### 4.3 Handle missing values
`unit_price` and `quantity` have 31 nulls each; these rows can't be used to compute `total`, so they are dropped.

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna().reset_index(drop=True)
print(df.isnull().sum().sum(), 'missing values left')
df.shape

### 4.4 Fix data types
- `unit_price`: `'$74.69'` (text) → `74.69` (float)
- `quantity`: float → int
- `date`: text (`dd/mm/yy`) → datetime
- `time`: text → time

In [ ]:
# NOTE: do NOT call .astype(float) before removing '$' – it raises a ValueError.
df['unit_price'] = (
    df['unit_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

df['quantity'] = df['quantity'].astype(int)

df['date'] = pd.to_datetime(df['date'], format='%d/%m/%y')
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.time

df.dtypes

## 5. Feature Engineering
`total = unit_price × quantity` (rounded to 2 decimals).

In [ ]:
df['total'] = (df['unit_price'] * df['quantity']).round(2)
df.head()

### Final check

In [ ]:
df.info()
df.shape

## 6. Save Clean Data to CSV

In [ ]:
df.to_csv('walmart_clean_data.csv', index=False)

## 7. Load into Databases

> ⚠️ **Never hard-code passwords in a notebook.** Credentials are read from environment variables below (fallbacks are for local testing only).
> `create_engine()` does **not** connect – the connection is only tested when `engine.connect()` is called.

### 7.1 MySQL

In [ ]:
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASS = quote_plus(os.getenv('MYSQL_PASS', ''))
MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = os.getenv('MYSQL_PORT', '3306')
MYSQL_DB   = os.getenv('MYSQL_DB', 'walmart_db')

engine_mysql = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASS}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)

try:
    with engine_mysql.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Connection to MySQL successful")
except Exception as e:
    print("Unable to connect to MySQL:", e)

In [ ]:
# 'replace' makes the cell safe to re-run (with 'append' every run duplicates the data)
rows = df.to_sql(name='walmart', con=engine_mysql, if_exists='replace', index=False)
print(rows, 'rows written to MySQL')

### 7.2 PostgreSQL

In [ ]:
PG_USER = os.getenv('PG_USER', 'postgres')
PG_PASS = quote_plus(os.getenv('PG_PASS', ''))
PG_HOST = os.getenv('PG_HOST', 'localhost')
PG_PORT = os.getenv('PG_PORT', '5432')
PG_DB   = os.getenv('PG_DB', 'walmart_db')

engine_psql = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

try:
    with engine_psql.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Connection to PostgreSQL successful")
except Exception as e:
    print("Unable to connect to PostgreSQL:", e)

In [ ]:
rows = df.to_sql(name='walmart', con=engine_psql, if_exists='replace', index=False)
print(rows, 'rows written to PostgreSQL')   # expected: 9969

### 7.3 Verify row counts

In [ ]:
print('MySQL     :', pd.read_sql('SELECT COUNT(*) AS n FROM walmart', engine_mysql)['n'][0])
print('PostgreSQL:', pd.read_sql('SELECT COUNT(*) AS n FROM walmart', engine_psql)['n'][0])
print('DataFrame :', len(df))